<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
Supplementary code for the <a href="https://mng.bz/lZ5B">Build a Reasoning Model (From Scratch)</a> book by <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>Code repository: <a href="https://github.com/rasbt/reasoning-from-scratch">https://github.com/rasbt/reasoning-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="https://mng.bz/lZ5B"><img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>


# Bölüm 5: Alıştırma Çözümleri

Bu not defterinde kullanılan paketler:

In [1]:
from importlib.metadata import version

used_libraries = [
    "reasoning_from_scratch",
    "torch",
    "tokenizers"  # Used by reasoning_from_scratch
]

for lib in used_libraries:
    print(f"{lib} version: {version(lib)}")

reasoning_from_scratch version: 0.1.13
torch version: 2.10.0
tokenizers version: 0.22.2


&nbsp;
## Alıştırma 5.1: Sezgisel puanlayıcıyı öz tutarlılıkta eşitlik bozucu olarak kullanmak

- Bunu uygulamanın birçok yolu var
- Belki de en kolayı, bunu öz tutarlılık fonksiyonunun dışında ele almak ve döndürülen sözlükle çalışmaktır (ör. eşitlik bozmayı doğrudan `evaluate_math500_stream` fonksiyonuna eklediğimiz 4.4 alıştırmasında yaptığımıza benzer şekilde)
- İlgili satırlar aşağıda gösterilmiştir

```python
# ...
from pathlib import Path
import time

from reasoning_from_scratch.ch05 import heuristic_score


def evaluate_math500_stream(
    model,
    tokenizer,
    device,
    math_data,
    out_path=None,
    max_new_tokens=2048,
    verbose=False,
    prompt_suffix="",
    temperature=1.0,
    top_p=1.0,
    seed=None,
    num_samples=10,
):
    if out_path is None:
        dev_name = str(device).replace(":", "-")
        out_path = Path(f"math500-{dev_name}.jsonl")

    num_examples = len(math_data)
    num_correct = 0
    start_time = time.time()

    with open(out_path, "w", encoding="utf-8") as f:
        for i, row in enumerate(math_data, start=1):
            prompt = render_prompt(row["problem"]) + prompt_suffix

            results = self_consistency_vote(
                model=model,
                tokenizer=tokenizer,
                prompt=prompt,
                device=device,
                num_samples=num_samples,
                temperature=temperature,
                top_p=top_p,
                max_new_tokens=max_new_tokens,
                show_progress=False,
                show_long_answer=False,
                seed=seed,
            )

            # Majority vote winner available
            if results["final_answer"] is not None:
                extracted = results["final_answer"]

            ### NEW: Break tie with heuristic_score
            else:
                best = None
                best_score = float("-inf")
            
                for cand in results["majority_winners"]:
                    scores = [
                        heuristic_score(results["full_answers"][idx], prompt=prompt)
                        for idx in results["groups"][cand]
                    ]
            
                    score = max(scores)
            
                    if score > best_score:
                        best_score = score
                        best = cand
            
                extracted = best

            # ...

    # ...
    return num_correct, num_examples, acc
```

- 3. bölümdeki temel çizgiye ve 4. bölümdeki öz tutarlılığa göre iyileşmeler aşağıda gösterilmiştir

|   | Yöntem                                   | Model | Doğruluk | Süre      |
|---|------------------------------------------|-------|----------|-----------|
| 1 | CoT istemli 4. bölüm temel çizgisi       | Temel | %33.4    | 129.2 dk  |
| 2 | Öz tutarlılık (n=3) + çoğunluk oyu       | Temel | %43.2    | 328.2 dk  |
| 3 | Öz tutarlılık (n=3) + sezgisel           | Temel | %43.4    | 326.5 dk  |
| 4 | Öz tutarlılık (n=3) + ort. logprob       | Temel | %44.8    | 327.7 dk  |

- Tablodaki doğruluk değerleri ve çalışma süreleri, MATH-500 test kümesindeki 500 örneğin tamamı üzerinde bir "cuda" GPU (DGX Spark) ile hesaplanmıştır

- Kolaylık olsun diye, [../02_math500-more-inference-scaling-scripts](../02_math500-more-inference-scaling-scripts) klasöründeki [self_consistency_scorer_math500.py](../02_math500-more-inference-scaling-scripts/self_consistency_scorer_math500.py) betiğini çalıştırabilirsiniz

- Ancak [#159](https://github.com/rasbt/reasoning-from-scratch/issues/159) numaralı sorunda tartışıldığı gibi, çoğunluk kazananını sezgisel puana göre belirlediğimizi, ama her çoğunluk çiftindeki yalnızca ilk örneği dikkate aldığımızı unutmayın
- Örneğin

&nbsp;
## Alıştırma 5.2: Sezgisel puanlayıcıyı N-en-iyi (Best-of-N) kurulumunda kullanmak

- N-en-iyi, birden çok yanıt üretmemiz açısından öz tutarlılığa benzer
- Ancak nihai yanıtı çoğunluk oyuna göre seçmek yerine, tüm yanıtları bir puanlama fonksiyonuyla (ör. `heuristic_score`) puanlayıp en yüksek puanlı yanıtı döndürürüz
- Bu davranışı uygulamanın birkaç yolu vardır; ancak en kolayı herhalde 4. bölümdeki mevcut öz tutarlılık fonksiyonunu şablon olarak kullanıp aşağıda gösterildiği gibi `heuristic_score` fonksiyonunu yerine takmaktır

```python
# ...

from reasoning_from_scratch.ch05 import (
    heuristic_score
)

def self_consistency_vote(
    model,
    tokenizer,
    prompt,
    device,
    num_samples=10,
    temperature=0.8,
    top_p=0.9,
    max_new_tokens=2048,
    show_progress=True,
    show_long_answer=False,
    seed=None,
):
    full_answers, short_answers = [], []
    counts = Counter()
    groups = {}
    majority_winners, final_answer = [], None
    best_score, best_idx = float("-inf"), None

    for i in range(num_samples):
        if seed is not None:
            torch.manual_seed(seed + i + 1)

        answer = generate_text_stream_concat_flex(
            model=model,
            tokenizer=tokenizer,
            prompt=prompt,
            device=device,
            max_new_tokens=max_new_tokens,
            verbose=show_long_answer,
            generate_func=generate_text_top_p_stream_cache,
            temperature=temperature,
            top_p=top_p,
        )

        short = extract_final_candidate(answer, fallback="number_then_full")
        full_answers.append(answer)
        short_answers.append(short)
        counts[short] += 1

        if short in groups:
            groups[short].append(i)
        else:
            groups[short] = [i]

        score = heuristic_score(answer, prompt=prompt)

        if score > best_score:
            best_score, best_idx = score, i

        if show_progress:
            print(f"[Sample {i+1}/{num_samples}] → {short!r}")

    if best_idx is not None:
        final_answer = short_answers[best_idx]
        majority_winners = [final_answer]

    return {
        "full_answers": full_answers,
        "short_answers": short_answers,
        "counts": dict(counts),
        "groups": groups,
        "majority_winners": majority_winners,
        "final_answer": final_answer,
    }

```

- Sonuçlar aşağıda gösterilmiştir

|   | Yöntem                                   | Model | Doğruluk | Süre      |
|---|------------------------------------------|-------|----------|-----------|
| 1 | Düşünce zinciri istemli temel çizgi      | Temel | %33.4    | 129.2 dk  |
| 2 | N-en-iyi (n=3) + sezgisel                | Temel | %40.6    | 327.7 dk  |
| 3 | N-en-iyi (n=3) + ort. logprob            | Temel | %43.2    | 330.2 dk  |

- Tablodaki doğruluk değerleri ve çalışma süreleri, MATH-500 test kümesindeki 500 örneğin tamamı üzerinde bir "cuda" GPU (DGX Spark) ile hesaplanmıştır

- Kolaylık olsun diye, [../02_math500-more-inference-scaling-scripts](../02_math500-more-inference-scaling-scripts) klasöründeki [self_consistency_scorer_math500.py](../02_math500-more-inference-scaling-scripts/best_of_n_math500.py) betiğini çalıştırabilirsiniz

&nbsp;
## Alıştırma 5.3: Logprob puanlayıcısını öz tutarlılıkta eşitlik bozucu olarak kullanmak

- Kod, `heuristic_score` yerine `avg_logprob_answer` kullanmamız dışında 5.1 alıştırmasındakine benzer

```python
# ...
# from reasoning_from_scratch.ch05 import heuristic_score
from reasoning_from_scratch.ch05 import avg_logprob_answer


def evaluate_math500_stream(
    model,
    tokenizer,
    device,
    math_data,
    out_path=None,
    max_new_tokens=2048,
    verbose=False,
    prompt_suffix="",
    temperature=1.0,
    top_p=1.0,
    seed=None,
    num_samples=10,
):
    if out_path is None:
        dev_name = str(device).replace(":", "-")
        out_path = Path(f"math500-{dev_name}.jsonl")

    num_examples = len(math_data)
    num_correct = 0
    start_time = time.time()

    with open(out_path, "w", encoding="utf-8") as f:
        for i, row in enumerate(math_data, start=1):
            prompt = render_prompt(row["problem"]) + prompt_suffix

            results = self_consistency_vote(
                model=model,
                tokenizer=tokenizer,
                prompt=prompt,
                device=device,
                num_samples=num_samples,
                temperature=temperature,
                top_p=top_p,
                max_new_tokens=max_new_tokens,
                show_progress=False,
                show_long_answer=False,
                seed=seed,
            )

            # Majority vote winner available
            if results["final_answer"] is not None:
                extracted = results["final_answer"]

            ### NEW: Break tie with avg_logprob_answer
            else:
                best = None
                best_score = float("-inf")
            
                # Consider all members of each majority group
                for cand in results["majority_winners"]:
                    scores = []
            
                    for idx in results["groups"][cand]:
                        candidate_full = results["full_answers"][idx]
            
                        score = avg_logprob_answer(
                            model=model,
                            tokenizer=tokenizer,
                            prompt=prompt,
                            answer=candidate_full,
                            device=device,
                        )
                        scores.append(score)
            
                    cand_score = max(scores)
            
                    if cand_score > best_score:
                        best_score = cand_score
                        best = cand
            
                extracted = best
            # ...

    # ...
    return num_correct, num_examples, acc
```

- 3. bölümdeki temel çizgiye ve 4. bölümdeki öz tutarlılığa göre iyileşmeler aşağıda gösterilmiştir

|   | Yöntem                                   | Model | Doğruluk | Süre      |
|---|------------------------------------------|-------|----------|-----------|
| 1 | Düşünce zinciri istemli temel çizgi      | Temel | %33.4    | 129.2 dk  |
| 2 | Öz tutarlılık (n=3) + çoğunluk oyu       | Temel | %43.2    | 328.2 dk  |
| 3 | Öz tutarlılık (n=3) + sezgisel           | Temel | %43.4    | 326.5 dk  |
| 4 | Öz tutarlılık (n=3) + ort logprob        | Temel | %44.8    | 327.7 dk  |

- Tablodaki doğruluk değerleri ve çalışma süreleri, MATH-500 test kümesindeki 500 örneğin tamamı üzerinde bir "cuda" GPU (DGX Spark) ile hesaplanmıştır

- Kolaylık olsun diye, [../02_math500-more-inference-scaling-scripts](../02_math500-more-inference-scaling-scripts) klasöründeki [self_consistency_scorer_math500.py](../02_math500-more-inference-scaling-scripts/best_of_n_math500.py) betiğini çalıştırabilirsiniz

&nbsp;
## Alıştırma 5.4: Logprob puanlayıcısını N-en-iyi (Best-of-N) kurulumunda kullanmak

- Logprob puanlayıcısıyla N-en-iyi uygulamak için 5.2 alıştırmasındaki kodu kullanabilir ve `heuristic_score` yerine `avg_logprob_answer` koyabiliriz:

```python

from reasoning_from_scratch.ch05 import (
    avg_logprob_answer
)


def self_consistency_vote(
    model,
    tokenizer,
    prompt,
    device,
    num_samples=10,
    temperature=0.8,
    top_p=0.9,
    max_new_tokens=2048,
    show_progress=True,
    show_long_answer=False,
    seed=None,
):
    full_answers, short_answers = [], []
    counts = Counter()
    groups = {}
    majority_winners, final_answer = [], None
    best_score, best_idx = float("-inf"), None

    for i in range(num_samples):
        if seed is not None:
            torch.manual_seed(seed + i + 1)

        answer = generate_text_stream_concat_flex(
            model=model,
            tokenizer=tokenizer,
            prompt=prompt,
            device=device,
            max_new_tokens=max_new_tokens,
            verbose=show_long_answer,
            generate_func=generate_text_top_p_stream_cache,
            temperature=temperature,
            top_p=top_p,
        )

        short = extract_final_candidate(answer, fallback="number_then_full")
        full_answers.append(answer)
        short_answers.append(short)
        counts[short] += 1

        if short in groups:
            groups[short].append(i)
        else:
            groups[short] = [i]

            score = avg_logprob_answer(
                model=model,
                tokenizer=tokenizer,
                prompt=prompt,
                answer=answer,
                device=device
            )
        if score > best_score:
            best_score, best_idx = score, i

        if show_progress:
            print(f"[Sample {i+1}/{num_samples}] → {short!r}")

    if best_idx is not None:
        final_answer = short_answers[best_idx]
        majority_winners = [final_answer]

    return {
        "full_answers": full_answers,
        "short_answers": short_answers,
        "counts": dict(counts),
        "groups": groups,
        "majority_winners": majority_winners,
        "final_answer": final_answer,
    }
```

- Sonuçlar aşağıda gösterilmiştir

| # | Yöntem                                   | Model | Doğruluk | Süre      |
|---|------------------------------------------|-------|----------|-----------|
| 1 | Düşünce zinciri istemli temel çizgi      | Temel | %33.4    | 129.2 dk  |
| 2 | N-en-iyi (n=3) + sezgisel                | Temel | TBD      | TBD       |
| 3 | N-en-iyi (n=3) + ort. logprob            | Temel | TBD      | TBD       |

- Tablodaki doğruluk değerleri ve çalışma süreleri, MATH-500 test kümesindeki 500 örneğin tamamı üzerinde bir "cuda" GPU (DGX Spark) ile hesaplanmıştır

- Kolaylık olsun diye, [../02_math500-more-inference-scaling-scripts](../02_math500-more-inference-scaling-scripts) klasöründeki [self_consistency_scorer_math500.py](../02_math500-more-inference-scaling-scripts/best_of_n_math500.py) betiğini çalıştırabilirsiniz

&nbsp;
## Alıştırma 5.5: Öz iyileştirme için sezgisel puanı kullanmak

- `heuristic_score` kullanmak aslında logprob puanını kullanmaktan bile daha basittir; tek yapmamız gereken aşağıdaki kodu değiştirmek:

```python
from functools import partial

avg_logprob_score = partial(
    avg_logprob_answer,
    model=model,
    tokenizer=tokenizer,
    device=device
)


torch.manual_seed(0)

results_logprob = self_refinement_loop(
    model=model,
    tokenizer=tokenizer,
    raw_prompt=raw_prompt,
    device=device,
    iterations=2,
    max_response_tokens=2048,
    max_critique_tokens=256,
    score_fn=avg_logprob_score,
    verbose=True,
    temperature=0.7,
    top_p=0.9,
)
```

- Güncellenen kod:

```python
torch.manual_seed(0)

results_logprob = self_refinement_loop(
    model=model,
    tokenizer=tokenizer,
    raw_prompt=raw_prompt,
    device=device,
    iterations=2,
    max_response_tokens=2048,
    max_critique_tokens=256,
    score_fn=heuristic_score,  # NEW
    verbose=True,
    temperature=0.7,
    top_p=0.9,
)
```

- Sezgisel puanlayıcı kullanılarak elde edilen sonuçlar 4., 5. ve 10. satırlarda gösterilmiştir:

|    | Yöntem                 | Puanlama      | Yineleme   | Model      | Doğruluk | Süre      |
|----|------------------------|---------------|------------|------------|----------|-----------|
| 1  | Temel çizgi (3. bölüm) | -             | -          | Temel      | %15.2    | 10.1 dk   |
| 2  | Öz iyileştirme         | Yok           | 1          | Temel      | %25.0    | 84.8 dk   |
| 3  | Öz iyileştirme         | Yok           | 2          | Temel      | %22.0    | 165.4 dk  |
| 4  | Öz iyileştirme         | Sezgisel      | 1          | Temel      | %21.6    | 84.7 dk   |
| 5  | Öz iyileştirme         | Sezgisel      | 2          | Temel      | %20.8    | 151.4 dk  |
| 6  | Öz iyileştirme         | Ort. logprob  | 1          | Temel      | %21.4    | 85.3 dk   |
| 7  | Öz iyileştirme         | Ort. logprob  | 2          | Temel      | %22.0    | 165.3 dk  |
|    |                        |               |            |            |          |           |
| 8  | Temel çizgi (3. bölüm) | -             | -          | Akıl yür.  | %48.2    | 182.1 dk  |
| 9  | Öz iyileştirme         | Yok           | 1          | Akıl yür.  | %56.6    | 498.8 dk  |
| 10 | Öz iyileştirme         | Sezgisel      | 1          | Akıl yür.  | %57.8    | 498.6 dk  |
| 11 | Öz iyileştirme         | Ort. logprob  | 1          | Akıl yür.  | %48.4    | 499.7 dk  |

- Tablodaki doğruluk değerleri ve çalışma süreleri, MATH-500 test kümesindeki 500 örneğin tamamı üzerinde bir "cuda" GPU (DGX Spark) ile hesaplanmıştır
- Kolaylık olsun diye, [../02_math500-more-inference-scaling-scripts](../02_math500-more-inference-scaling-scripts) klasöründeki [self_consistency_scorer_math500.py](../02_math500-more-inference-scaling-scripts/self_refinement_math500.py) betiğini çalıştırabilirsiniz